In [2]:
import os
import shutil

source_dir = "data/raw"          # folder where everything is currently mixed
images_out = "data/all_images"
labels_out = "data/all_labels"


In [3]:
images = {f.rsplit(".", 1)[0] for f in os.listdir(images_out)}
labels = {f.rsplit(".", 1)[0] for f in os.listdir(labels_out)}

missing_labels = images - labels   # images with no txt file
missing_images = labels - images   # txt files with no matching image

print(f"Images missing labels: {len(missing_labels)}")
print(f"Labels missing images: {len(missing_images)}")

if missing_labels:
    print("Example:", list(missing_labels)[:5])
if missing_images:
    print("Example:", list(missing_images)[:5])


prefix_to_class = {
    "speedbreaker": 0,   # adjust these prefixes to match your actual filenames
    "pothole": 1,
    "unpaved": 2,
}

mismatches = []
for fname in os.listdir(labels_out):
    name_lower = fname.lower()
    expected_class = None
    for prefix, cls_id in prefix_to_class.items():
        if prefix in name_lower:
            expected_class = cls_id
            break

    if expected_class is None:
        continue  # filename didn't match any known prefix, skip check

    with open(os.path.join(labels_out, fname)) as f:
        for line in f:
            actual_class = int(line.split()[0])
            if actual_class != expected_class:
                mismatches.append((fname, actual_class, expected_class))

print(f"Found {len(mismatches)} mismatches")
print(mismatches[:10])

Images missing labels: 0
Labels missing images: 0
Found 454 mismatches
[('PotHoles_10.txt', 2, 1), ('PotHoles_100.txt', 2, 1), ('PotHoles_104.txt', 2, 1), ('PotHoles_105.txt', 2, 1), ('PotHoles_106.txt', 2, 1), ('PotHoles_107.txt', 2, 1), ('PotHoles_109.txt', 0, 1), ('PotHoles_11.txt', 2, 1), ('PotHoles_110.txt', 2, 1), ('PotHoles_117.txt', 2, 1)]


In [4]:
from collections import Counter

confusion = Counter()   # (expected_class, actual_class) -> count

for fname in os.listdir(labels_out):
    name_lower = fname.lower()
    expected_class = None
    for prefix, cls_id in prefix_to_class.items():
        if prefix in name_lower:
            expected_class = cls_id
            break
    if expected_class is None:
        continue

    with open(os.path.join(labels_out, fname)) as f:
        for line in f:
            actual_class = int(line.split()[0])
            confusion[(expected_class, actual_class)] += 1

for (exp, act), count in sorted(confusion.items(), key=lambda x: -x[1]):
    print(f"Expected {exp}, Actual {act}: {count} lines")

import os

sample_files = os.listdir(labels_out)[:4000]
prefixes_found = set()
for f in sample_files:
    # print the part before the last underscore+number
    prefixes_found.add(f.rsplit("_", 1)[0])

print(prefixes_found)

import re

class_keywords = {
    0: ["sb_", "speed_", "unmarkedbump"],       # speed breaker
    1: ["pothole", "pot_holes"],                 # pothole
    2: ["unpaved", "ungradedroad", "unpavedroad"], # unpaved road
}

Expected 2, Actual 2: 1913 lines
Expected 1, Actual 1: 1222 lines
Expected 1, Actual 2: 272 lines
Expected 1, Actual 0: 128 lines
Expected 2, Actual 0: 27 lines
Expected 2, Actual 1: 27 lines
{'frame+1888', 'frame+1152', 'frame+341.txt', 'UnMarkedBump', 'frame+688.txt', 'frame+1438', 'frame+79.txt', 'frame+602.txt', 'frame+535.txt', 'frame+337.txt', 'frame+759', 'frame+659', 'frame+1644', 'frame+6', 'frame+50.txt', 'frame+1145', 'frame+442.txt', 'frame+662.txt', 'frame+128.txt', 'frame+428.txt', 'frame+1424', 'frame+353.txt', 'frame+381.txt', 'frame+496.txt', 'frame+501.txt', 'frame+53.txt', 'frame+175', 'frame+261.txt', 'frame+515.txt', 'frame+1409', 'frame+687.txt', 'frame+310.txt', 'frame+737.txt', 'frame+244.txt', 'frame+7', 'frame+82.txt', 'frame+98.txt', 'frame+734.txt', 'frame+80.txt', 'frame+275.txt', 'frame+348.txt', 'frame+446', 'frame+346.txt', 'frame+732.txt', 'frame+423.txt', 'frame+471.txt', 'frame+345.txt', 'frame+1428', 'frame+1540.txt', 'frame+707.txt', 'frame+268.txt'

In [5]:
def get_expected_classes(fname):
    """Returns a set of expected class ids based on filename, or None if unrecognized."""
    name_lower = fname.lower()

    if name_lower.startswith("frame+") or name_lower.startswith("frame_"):
        return None   # no class info in filename — can't check these

    found = set()
    for cls_id, keywords in class_keywords.items():
        if any(kw in name_lower for kw in keywords):
            found.add(cls_id)

    return found if found else None

In [6]:
from collections import Counter

confusion = Counter()
unrecognized_files = []

for fname in os.listdir(labels_out):
    expected = get_expected_classes(fname)
    if expected is None:
        unrecognized_files.append(fname)
        continue

    with open(os.path.join(labels_out, fname)) as f:
        for line in f:
            actual_class = int(line.split()[0])
            if actual_class not in expected:
                confusion[(tuple(sorted(expected)), actual_class)] += 1

print(f"Unrecognized (frame+ etc.) files: {len(unrecognized_files)}")
for (exp, act), count in sorted(confusion.items(), key=lambda x: -x[1]):
    print(f"Expected {exp}, Actual {act}: {count} lines")


Unrecognized (frame+ etc.) files: 539
Expected (1,), Actual 2: 214 lines
Expected (2,), Actual 1: 137 lines
Expected (1,), Actual 0: 128 lines
Expected (2,), Actual 0: 103 lines
Expected (0,), Actual 1: 25 lines


In [7]:
import re
from collections import Counter, defaultdict

def get_source_tag(fname):
    """Extract the naming pattern before the trailing number, to identify the source dataset."""
    name = fname.rsplit(".", 1)[0]
    # strip trailing _<number> or +<number>
    tag = re.sub(r'[_+]\d+$', '', name)
    return tag

source_class_dist = defaultdict(Counter)

for fname in os.listdir(labels_out):
    tag = get_source_tag(fname)
    with open(os.path.join(labels_out, fname)) as f:
        for line in f:
            cls_id = int(line.split()[0])
            source_class_dist[tag][cls_id] += 1

for tag, dist in sorted(source_class_dist.items()):
    total = sum(dist.values())
    print(f"{tag} (total {total} lines): {dict(dist)}")

AN_unpaved (total 597 lines): {2: 597}
PotHoles (total 1387 lines): {1: 1047, 2: 212, 0: 128}
PotHoles_0 - Copy - Copy (total 2 lines): {1: 2}
PotHoles_551 - Copy (total 1 lines): {2: 1}
PotHoles_554 - Copy (total 2 lines): {2: 1, 1: 1}
PotHoles_592 - Copy (total 1 lines): {1: 1}
PotHoles_plus_unpaved (total 229 lines): {1: 171, 2: 58}
SB_ (total 246 lines): {0: 246}
Speed (total 257 lines): {1: 25, 0: 232}
UnMarkedBump (total 466 lines): {0: 466}
UnPavedRoad_ (total 1370 lines): {2: 1316, 0: 27, 1: 27}
UngradedRoad (total 272 lines): {0: 76, 2: 86, 1: 110}
frame (total 476 lines): {2: 417, 1: 59}
frame+10_ (total 1 lines): {2: 1}
frame+1137_ (total 1 lines): {1: 1}
frame+1138_ (total 1 lines): {1: 1}
frame+1139_ (total 1 lines): {1: 1}
frame+1140_ (total 1 lines): {1: 1}
frame+1142_ (total 1 lines): {1: 1}
frame+1143_ (total 1 lines): {1: 1}
frame+1144_ (total 1 lines): {1: 1}
frame+1145_ (total 1 lines): {1: 1}
frame+1146_ (total 1 lines): {1: 1}
frame+1147_ (total 1 lines): {1: 1}
f

In [8]:
import os

def fix_an_unpaved_labels(labels_dir):
    fixed_count = 0
    for fname in os.listdir(labels_dir):
        if not fname.lower().startswith("an_unpaved"):
            continue

        path = os.path.join(labels_dir, fname)
        new_lines = []
        with open(path) as f:
            for line in f:
                parts = line.split()
                cls_id = int(parts[0])
                if cls_id == 1:
                    cls_id = 2   # remap: their "1" = unpaved -> our "2"
                new_lines.append(f"{cls_id} {' '.join(parts[1:])}")
                fixed_count += 1

        with open(path, "w") as f:
            f.write("\n".join(new_lines) + "\n")

    print(f"Remapped {fixed_count} lines across AN_unpaved files")

fix_an_unpaved_labels(labels_out)

Remapped 597 lines across AN_unpaved files


In [9]:
import hashlib
from collections import defaultdict

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

hash_to_files = defaultdict(list)
for fname in os.listdir(images_out):
    full_path = os.path.join(images_out, fname)
    hash_to_files[file_hash(full_path)].append(fname)

duplicates = {h: files for h, files in hash_to_files.items() if len(files) > 1}
print(f"Found {len(duplicates)} sets of duplicate images")
for h, files in list(duplicates.items())[:10]:
    print(files)

Found 0 sets of duplicate images


In [10]:
def get_source_tag(fname):
    name = fname.rsplit(".", 1)[0]
    # strip trailing _<number> or +<number> or +<number>_ (frame+N_ pattern)
    tag = re.sub(r'[_+]\d+_?$', '', name)
    return tag
get_source_tag(fname)

'UnPavedRoad_'

In [11]:
import os
import hashlib
from collections import Counter, defaultdict
from PIL import Image

IMAGES_DIR = "data/all_images"
LABELS_DIR = "data/all_labels"
VALID_CLASSES = {0, 1, 2}   # speed_breaker, pothole, unpaved_road

report = defaultdict(list)

# ---------- 1. Pairing check ----------
image_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(IMAGES_DIR)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))}
label_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(LABELS_DIR)
               if f.endswith(".txt")}

images_without_labels = set(image_files) - set(label_files)
labels_without_images = set(label_files) - set(image_files)

print(f"Total images: {len(image_files)}")
print(f"Total labels: {len(label_files)}")
print(f"Images missing labels: {len(images_without_labels)}")
print(f"Labels missing images: {len(labels_without_images)}")

if images_without_labels:
    report["missing_labels"] = list(images_without_labels)[:10]
if labels_without_images:
    report["missing_images"] = list(labels_without_images)[:10]

# ---------- 2. Corrupt / unreadable image check ----------
corrupt_images = []
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        corrupt_images.append((fname, str(e)))

print(f"\nCorrupt/unreadable images: {len(corrupt_images)}")
if corrupt_images:
    report["corrupt_images"] = corrupt_images[:10]

# ---------- 3. Label content validation ----------
empty_labels = []
malformed_lines = []
invalid_class_ids = []
out_of_range_coords = []
class_counts = Counter()

for name, fname in label_files.items():
    path = os.path.join(LABELS_DIR, fname)
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]

    if len(lines) == 0:
        empty_labels.append(fname)
        continue

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            malformed_lines.append((fname, line))
            continue

        cls_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        if cls_id not in VALID_CLASSES:
            invalid_class_ids.append((fname, cls_id))

        if not all(0.0 <= c <= 1.0 for c in coords):
            out_of_range_coords.append((fname, coords))

        class_counts[cls_id] += 1

print(f"\nEmpty label files: {len(empty_labels)}")
print(f"Malformed lines (not 5 values): {len(malformed_lines)}")
print(f"Invalid class IDs (outside {VALID_CLASSES}): {len(invalid_class_ids)}")
print(f"Coordinates outside 0-1 range (likely pixel coords, not normalized): {len(out_of_range_coords)}")
print(f"\nClass distribution: {dict(class_counts)}")

if empty_labels: report["empty_labels"] = empty_labels[:10]
if malformed_lines: report["malformed_lines"] = malformed_lines[:10]
if invalid_class_ids: report["invalid_class_ids"] = invalid_class_ids[:10]
if out_of_range_coords: report["out_of_range_coords"] = out_of_range_coords[:10]

# ---------- 4. Duplicate image check (hash-based) ----------
hash_to_files = defaultdict(list)
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    with open(path, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    hash_to_files[h].append(fname)

duplicate_groups = {h: files for h, files in hash_to_files.items() if len(files) > 1}
total_duplicate_files = sum(len(v) - 1 for v in duplicate_groups.values())

print(f"\nDuplicate image groups: {len(duplicate_groups)}")
print(f"Total redundant duplicate files (excess copies): {total_duplicate_files}")
if duplicate_groups:
    report["duplicate_groups"] = list(duplicate_groups.items())[:5]

# ---------- 5. Final verdict ----------
print("\n" + "="*50)
if not any(report.values()):
    print("✅ Dataset looks clean. Ready for train/val/test split.")
else:
    print("⚠️  Issues found — review 'report' dict before splitting.")
    for k in report:
        if report[k]:
            print(f"  - {k}: {len(report[k])} shown (sample above)")

Total images: 4306
Total labels: 4306
Images missing labels: 0
Labels missing images: 0

Corrupt/unreadable images: 0

Empty label files: 0
Malformed lines (not 5 values): 0
Invalid class IDs (outside {0, 1, 2}): 0
Coordinates outside 0-1 range (likely pixel coords, not normalized): 0

Class distribution: {2: 2757, 1: 1499, 0: 1175}

Duplicate image groups: 0
Total redundant duplicate files (excess copies): 0

✅ Dataset looks clean. Ready for train/val/test split.


In [12]:
import os

# Re-run hash grouping (or reuse hash_to_files from the verification script)
duplicates_to_review = []

for h, files in hash_to_files.items():
    if len(files) <= 1:
        continue

    # Check if their label files are also identical
    label_contents = []
    for fname in files:
        label_path = os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt")
        with open(label_path) as f:
            label_contents.append(f.read().strip())

    all_labels_same = len(set(label_contents)) == 1
    duplicates_to_review.append((files, all_labels_same))

# Split into safe-to-auto-delete vs needs-manual-check
safe_duplicates = [d for d in duplicates_to_review if d[1]]
conflicting_duplicates = [d for d in duplicates_to_review if not d[1]]

print(f"Duplicate groups with identical labels (safe to auto-dedupe): {len(safe_duplicates)}")
print(f"Duplicate groups with DIFFERENT labels (needs manual look): {len(conflicting_duplicates)}")
for files, _ in conflicting_duplicates[:5]:
    print(files)

Duplicate groups with identical labels (safe to auto-dedupe): 0
Duplicate groups with DIFFERENT labels (needs manual look): 0


In [13]:
deleted_count = 0
for h, files in hash_to_files.items():
    if len(files) <= 1:
        continue
    label_contents = []
    for fname in files:
        label_path = os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt")
        with open(label_path) as f:
            label_contents.append(f.read().strip())
    if len(set(label_contents)) != 1:
        continue   # skip conflicting ones — handle manually

    keep, *remove = files
    for fname in remove:
        os.remove(os.path.join(IMAGES_DIR, fname))
        os.remove(os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt"))
        deleted_count += 1

print(f"Deleted {deleted_count} redundant duplicate files.")

Deleted 0 redundant duplicate files.


In [14]:
import os
import hashlib
from collections import Counter, defaultdict
from PIL import Image

IMAGES_DIR = "data/all_images"
LABELS_DIR = "data/all_labels"
VALID_CLASSES = {0, 1, 2}   # speed_breaker, pothole, unpaved_road

report = defaultdict(list)

# ---------- 1. Pairing check ----------
image_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(IMAGES_DIR)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))}
label_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(LABELS_DIR)
               if f.endswith(".txt")}

images_without_labels = set(image_files) - set(label_files)
labels_without_images = set(label_files) - set(image_files)

print(f"Total images: {len(image_files)}")
print(f"Total labels: {len(label_files)}")
print(f"Images missing labels: {len(images_without_labels)}")
print(f"Labels missing images: {len(labels_without_images)}")

if images_without_labels:
    report["missing_labels"] = list(images_without_labels)[:10]
if labels_without_images:
    report["missing_images"] = list(labels_without_images)[:10]

# ---------- 2. Corrupt / unreadable image check ----------
corrupt_images = []
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        corrupt_images.append((fname, str(e)))

print(f"\nCorrupt/unreadable images: {len(corrupt_images)}")
if corrupt_images:
    report["corrupt_images"] = corrupt_images[:10]

# ---------- 3. Label content validation ----------
empty_labels = []
malformed_lines = []
invalid_class_ids = []
out_of_range_coords = []
class_counts = Counter()

for name, fname in label_files.items():
    path = os.path.join(LABELS_DIR, fname)
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]

    if len(lines) == 0:
        empty_labels.append(fname)
        continue

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            malformed_lines.append((fname, line))
            continue

        cls_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        if cls_id not in VALID_CLASSES:
            invalid_class_ids.append((fname, cls_id))

        if not all(0.0 <= c <= 1.0 for c in coords):
            out_of_range_coords.append((fname, coords))

        class_counts[cls_id] += 1

print(f"\nEmpty label files: {len(empty_labels)}")
print(f"Malformed lines (not 5 values): {len(malformed_lines)}")
print(f"Invalid class IDs (outside {VALID_CLASSES}): {len(invalid_class_ids)}")
print(f"Coordinates outside 0-1 range (likely pixel coords, not normalized): {len(out_of_range_coords)}")
print(f"\nClass distribution: {dict(class_counts)}")

if empty_labels: report["empty_labels"] = empty_labels[:10]
if malformed_lines: report["malformed_lines"] = malformed_lines[:10]
if invalid_class_ids: report["invalid_class_ids"] = invalid_class_ids[:10]
if out_of_range_coords: report["out_of_range_coords"] = out_of_range_coords[:10]

# ---------- 4. Duplicate image check (hash-based) ----------
hash_to_files = defaultdict(list)
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    with open(path, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    hash_to_files[h].append(fname)

duplicate_groups = {h: files for h, files in hash_to_files.items() if len(files) > 1}
total_duplicate_files = sum(len(v) - 1 for v in duplicate_groups.values())

print(f"\nDuplicate image groups: {len(duplicate_groups)}")
print(f"Total redundant duplicate files (excess copies): {total_duplicate_files}")
if duplicate_groups:
    report["duplicate_groups"] = list(duplicate_groups.items())[:5]

# ---------- 5. Final verdict ----------
print("\n" + "="*50)
if not any(report.values()):
    print("✅ Dataset looks clean. Ready for train/val/test split.")
else:
    print("⚠️  Issues found — review 'report' dict before splitting.")
    for k in report:
        if report[k]:
            print(f"  - {k}: {len(report[k])} shown (sample above)")

Total images: 4306
Total labels: 4306
Images missing labels: 0
Labels missing images: 0

Corrupt/unreadable images: 0

Empty label files: 0
Malformed lines (not 5 values): 0
Invalid class IDs (outside {0, 1, 2}): 0
Coordinates outside 0-1 range (likely pixel coords, not normalized): 0

Class distribution: {2: 2757, 1: 1499, 0: 1175}

Duplicate image groups: 0
Total redundant duplicate files (excess copies): 0

✅ Dataset looks clean. Ready for train/val/test split.


In [15]:
import os

def show_group_labels(files):
    for fname in files:
        label_path = os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt")
        with open(label_path) as f:
            content = f.read().strip()
        print(f"  {fname}:")
        print(f"    {content}\n")

# Peek at first 5 conflicting groups
for files, _ in conflicting_duplicates[:5]:
    print(files)
    show_group_labels(files)
    print("-" * 40)

In [16]:
def choose_best(files):
    def score(fname):
        label_path = os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt")
        with open(label_path) as f:
            num_boxes = len([l for l in f if l.strip()])
        is_copy = "copy" in fname.lower()
        return (num_boxes, not is_copy)   # more boxes wins; non-"Copy" wins on tie

    return max(files, key=score)

deleted_count = 0
kept_log = []

for files, _ in conflicting_duplicates:
    best = choose_best(files)
    kept_log.append((files, best))
    for fname in files:
        if fname == best:
            continue
        os.remove(os.path.join(IMAGES_DIR, fname))
        label_path = os.path.join(LABELS_DIR, fname.rsplit(".", 1)[0] + ".txt")
        if os.path.exists(label_path):
            os.remove(label_path)
        deleted_count += 1

print(f"Deleted {deleted_count} files, kept the best-annotated version from each group.")

# Print a sample of decisions made, for your sanity/report
for files, best in kept_log[:10]:
    print(f"Group {files} → kept {best}")

Deleted 0 files, kept the best-annotated version from each group.


In [17]:
import os
import hashlib
from collections import Counter, defaultdict
from PIL import Image

IMAGES_DIR = "data/all_images"
LABELS_DIR = "data/all_labels"
VALID_CLASSES = {0, 1, 2}   # speed_breaker, pothole, unpaved_road

report = defaultdict(list)

# ---------- 1. Pairing check ----------
image_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(IMAGES_DIR)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))}
label_files = {f.rsplit(".", 1)[0]: f for f in os.listdir(LABELS_DIR)
               if f.endswith(".txt")}

images_without_labels = set(image_files) - set(label_files)
labels_without_images = set(label_files) - set(image_files)

print(f"Total images: {len(image_files)}")
print(f"Total labels: {len(label_files)}")
print(f"Images missing labels: {len(images_without_labels)}")
print(f"Labels missing images: {len(labels_without_images)}")

if images_without_labels:
    report["missing_labels"] = list(images_without_labels)[:10]
if labels_without_images:
    report["missing_images"] = list(labels_without_images)[:10]

# ---------- 2. Corrupt / unreadable image check ----------
corrupt_images = []
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception as e:
        corrupt_images.append((fname, str(e)))

print(f"\nCorrupt/unreadable images: {len(corrupt_images)}")
if corrupt_images:
    report["corrupt_images"] = corrupt_images[:10]

# ---------- 3. Label content validation ----------
empty_labels = []
malformed_lines = []
invalid_class_ids = []
out_of_range_coords = []
class_counts = Counter()

for name, fname in label_files.items():
    path = os.path.join(LABELS_DIR, fname)
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip()]

    if len(lines) == 0:
        empty_labels.append(fname)
        continue

    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            malformed_lines.append((fname, line))
            continue

        cls_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        if cls_id not in VALID_CLASSES:
            invalid_class_ids.append((fname, cls_id))

        if not all(0.0 <= c <= 1.0 for c in coords):
            out_of_range_coords.append((fname, coords))

        class_counts[cls_id] += 1

print(f"\nEmpty label files: {len(empty_labels)}")
print(f"Malformed lines (not 5 values): {len(malformed_lines)}")
print(f"Invalid class IDs (outside {VALID_CLASSES}): {len(invalid_class_ids)}")
print(f"Coordinates outside 0-1 range (likely pixel coords, not normalized): {len(out_of_range_coords)}")
print(f"\nClass distribution: {dict(class_counts)}")

if empty_labels: report["empty_labels"] = empty_labels[:10]
if malformed_lines: report["malformed_lines"] = malformed_lines[:10]
if invalid_class_ids: report["invalid_class_ids"] = invalid_class_ids[:10]
if out_of_range_coords: report["out_of_range_coords"] = out_of_range_coords[:10]

# ---------- 4. Duplicate image check (hash-based) ----------
hash_to_files = defaultdict(list)
for name, fname in image_files.items():
    path = os.path.join(IMAGES_DIR, fname)
    with open(path, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()
    hash_to_files[h].append(fname)

duplicate_groups = {h: files for h, files in hash_to_files.items() if len(files) > 1}
total_duplicate_files = sum(len(v) - 1 for v in duplicate_groups.values())

print(f"\nDuplicate image groups: {len(duplicate_groups)}")
print(f"Total redundant duplicate files (excess copies): {total_duplicate_files}")
if duplicate_groups:
    report["duplicate_groups"] = list(duplicate_groups.items())[:5]

# ---------- 5. Final verdict ----------
print("\n" + "="*50)
if not any(report.values()):
    print("✅ Dataset looks clean. Ready for train/val/test split.")
else:
    print("⚠️  Issues found — review 'report' dict before splitting.")
    for k in report:
        if report[k]:
            print(f"  - {k}: {len(report[k])} shown (sample above)")

Total images: 4306
Total labels: 4306
Images missing labels: 0
Labels missing images: 0

Corrupt/unreadable images: 0

Empty label files: 0
Malformed lines (not 5 values): 0
Invalid class IDs (outside {0, 1, 2}): 0
Coordinates outside 0-1 range (likely pixel coords, not normalized): 0

Class distribution: {2: 2757, 1: 1499, 0: 1175}

Duplicate image groups: 0
Total redundant duplicate files (excess copies): 0

✅ Dataset looks clean. Ready for train/val/test split.


In [18]:
import random

random.seed(42)
paired = sorted(images & labels)  # only files that have both image+label
random.shuffle(paired)

n = len(paired)
train_set = paired[:int(.7*n)]
val_set   = paired[int(.7*n):int(.9*n)]
test_set  = paired[int(.9*n):]

for split, names in [("train", train_set), ("val", val_set), ("test", test_set)]:
    os.makedirs(f"data/images/{split}", exist_ok=True)
    os.makedirs(f"data/labels/{split}", exist_ok=True)
    for name in names:
        # handle .jpg or .jpeg extension
        for ext in [".jpg", ".jpeg"]:
            src_img = os.path.join(images_out, name + ext)
            if os.path.exists(src_img):
                shutil.copy(src_img, f"data/images/{split}/{name}{ext}")
                break
        shutil.copy(os.path.join(labels_out, name + ".txt"), f"data/labels/{split}/{name}.txt")

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Train: 3014, Val: 861, Test: 431
